In [48]:
import pandas as pd
import numpy as np

# 先讀進模擬資料
data = pd.read_csv(r"C:\Users\USER\Desktop\碩論\程式碼\embedding_data.csv")

cols = ["X2","X3","X1"] # 針對變數標準化，後面做softmax的時候，數值才不會爆掉

data[cols] = (data[cols] - data[cols].mean()) / data[cols].std()

data

,X2,X3,X1,Y,raw_x1,raw_x2,raw_x3
0,2.985660,-0.455945,0.902386,4.686122,0.650593,2.913862,-0.529439
1,2.183840,0.287470,0.274274,4.410815,0.162753,2.128470,0.232170
2,1.646134,-0.089969,1.195760,3.444338,0.878450,1.601779,-0.154506
3,1.815760,0.819613,-0.757854,3.378867,-0.638878,1.767930,0.777338
4,2.049346,-0.976188,-0.996825,3.135581,-0.824481,1.996731,-1.062414
...,...,...,...,...,...,...,...
95,-1.198872,-1.281204,1.311417,-2.831205,0.968278,-1.184944,-1.374896
96,-1.025966,-0.577673,1.518317,-3.037451,1.128972,-1.015579,-0.654146
97,-0.942290,-1.940348,-2.447304,-3.492096,-1.951035,-0.933618,-2.050172
98,-0.413190,-1.195821,0.346293,-3.672475,0.218689,-0.415357,-1.287423


In [ ]:
# 為了測試模型架構，先使用第一筆資料
test = data.iloc[0,:3]
y = data.iloc[0,3]

y = np.array(y) # 為了轉tensor，所以serious要轉成array

import torch

shape = (1,8) # 投影成8維，為了方便後續的attention計算

random = torch.randn(shape, requires_grad = True) # 需要學習的地方

test = torch.tensor(test, dtype = torch.float32) # tensor毛好多，統一設定dtype = float32

test = test.reshape(1,3)

y = torch.tensor(y, dtype = torch.float32)

random = random.reshape(-1,1) # 為了後續矩陣乘法的需要

# 將隨便猜的八維向量，根據原始的embedding縮放
# 目前是所有feature都用同一個random空間
# 可以考慮讓每個feature都各自有random空間(後續改動的地方)
E =  random @ test 

wq_shape = (4,8) # KQ space先設定4維，反正比embedding的小

wq = torch.randn(wq_shape, requires_grad = True) # 需要學習的地方

Q = wq @ E

wk_shape = (4,8)

wk = torch.randn(wq_shape, requires_grad = True) # 需要學習的地方

K = wk @ E

scores = K.T @ Q 

dk = 4 # QK space的維度

scores = scores/np.sqrt(dk)

import torch.nn.functional as F

attn = F.softmax(scores, dim=0)

wv_shape = (8,8) # 需要變回原本embedding的維度

wv = torch.randn(wv_shape, requires_grad = True) # 需要學習的地方

V = wv @ E

delta_E = V @ attn

New_E = E + delta_E

print("初始灰階embedding:")
print(test)
print("-----------------------------------------------------")
print("原本隨機的embedding:")
print(E)
print("-----------------------------------------------------")
print("經過shift之後:")
print(delta_E)

初始灰階embedding:
tensor([[ 2.9857, -0.4559,  0.9024]])
-----------------------------------------------------
原本隨機的embedding:
tensor([[ 3.5360, -0.5400,  1.0687],
        [-4.4270,  0.6761, -1.3380],
        [ 4.4899, -0.6857,  1.3570],
        [ 2.6848, -0.4100,  0.8115],
        [-2.3080,  0.3525, -0.6976],
        [ 0.8123, -0.1240,  0.2455],
        [-1.5874,  0.2424, -0.4798],
        [ 1.9528, -0.2982,  0.5902]], grad_fn=<MmBackward0>)
-----------------------------------------------------
經過shift之後:
tensor([[ -0.6961,   4.5577,  -0.6961],
        [ -0.9551,   6.2537,  -0.9551],
        [ -1.9450,  12.7355,  -1.9450],
        [ -1.9034,  12.4631,  -1.9034],
        [ -1.6392,  10.7334,  -1.6392],
        [  1.9671, -12.8798,   1.9670],
        [  1.8946, -12.4056,   1.8946],
        [ -0.6439,   4.2161,  -0.6439]], grad_fn=<MmBackward0>)


C:\Users\USER\AppData\Local\Temp\ipykernel_17872\3947416136.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  test = torch.tensor(test, dtype = torch.float32) # tensor毛好多，統一設定dtype = float32


In [50]:
print("新的embedding:")
print(New_E) # 新的embedding

新的embedding:
tensor([[  2.8399,   4.0177,   0.3727],
        [ -5.3821,   6.9297,  -2.2931],
        [  2.5449,  12.0499,  -0.5879],
        [  0.7814,  12.0531,  -1.0919],
        [ -3.9472,  11.0858,  -2.3368],
        [  2.7794, -13.0039,   2.2125],
        [  0.3073, -12.1632,   1.4148],
        [  1.3089,   3.9178,  -0.0537]], grad_fn=<AddBackward0>)


In [53]:
# y = 0.2*x1 + 1.5*x2 + 0.8*x3 可以從匯出模擬資料.ipynb得到正確的模型模擬
y_res = data["Y"] - (0.2 * data["raw_x1"] + 1.5 * data["raw_x2"] + 0.8 * data["raw_x3"])